In [21]:
import pyomo.environ as pe
import pyomo.opt as po
solver = po.SolverFactory('glpk')

In [22]:
model = pe.ConcreteModel()

In [23]:
model.x1 = pe.Var(domain=pe.Binary)
model.x2 = pe.Var(domain=pe.Binary)
model.x3 = pe.Var(domain=pe.Binary)
model.x4 = pe.Var(domain=pe.Binary)
model.x5 = pe.Var(domain=pe.Binary)



In [24]:
obj_expr = 3*model.x1 + 4*model.x2 + 5*model.x3 + 8*model.x4 + 9* model.x5
model.obj = pe.Objective(sense=pe.maximize, expr=obj_expr)


In [25]:
con_expr = 2 * model.x1+ 3*model.x2 + 4*model.x3+ 5*model.x4+ 9*model.x5 <=20
model.con=pe.Constraint(expr=con_expr)

In [26]:
result = solver.solve(model, tee=True)

GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --write C:\Users\valer\AppData\Local\Temp\tmpe4lu6fuz.glpk.raw --wglp C:\Users\valer\AppData\Local\Temp\tmpn50z5h2i.glpk.glp
 --cpxlp C:\Users\valer\AppData\Local\Temp\tmplu2b24b2.pyomo.lp
Reading problem data from 'C:\Users\valer\AppData\Local\Temp\tmplu2b24b2.pyomo.lp'...
C:\Users\valer\AppData\Local\Temp\tmplu2b24b2.pyomo.lp:28: warning: lower bound of variable 'x2' redefined
C:\Users\valer\AppData\Local\Temp\tmplu2b24b2.pyomo.lp:28: warning: upper bound of variable 'x2' redefined
1 row, 5 columns, 5 non-zeros
5 integer variables, all of which are binary
33 lines were read
Writing problem data to 'C:\Users\valer\AppData\Local\Temp\tmpn50z5h2i.glpk.glp'...
20 lines were written
GLPK Integer Optimizer, v4.65
1 row, 5 columns, 5 non-zeros
5 integer variables, all of which are binary
Preprocessing...
3 constraint coefficient(s) were reduced
1 row, 5 columns, 5 non-zeros
5 integer variables, all of which are b

In [27]:
print(pe.value(model.x1))
print(pe.value(model.x2))
print(pe.value(model.x3))
print(pe.value(model.x4))
print(pe.value(model.x5))

1.0
0.0
1.0
1.0
1.0


In [28]:
model.pprint()

5 Var Declarations
    x1 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   1.0 :     1 : False : False : Binary
    x2 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   0.0 :     1 : False : False : Binary
    x3 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   1.0 :     1 : False : False : Binary
    x4 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   1.0 :     1 : False : False : Binary
    x5 : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :   1.0 :     1 : False : False : Binary

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 3*x1 + 4*x2 + 5*x3 + 8*x4 + 9*x5

1 Constraint Declarations
    con : Si

In [29]:
model = pe.ConcreteModel()
model.N = pe.RangeSet(1 , 5)
print(set(model.N))

{1, 2, 3, 4, 5}


In [30]:
c = {1:3,2:4,3:5,4:8,5:9}
a = {1:2,2:3,3:4,4:5,5:9}
b=20

In [ ]:
model.c = pe.Param(model.N, initialize=c)
model.a = pe.Param(model.N,initialize=a)
model.b = pe.Param(initialize=b)


In [32]:
model.x=pe.Var(model.N,domain=pe.Binary)

In [33]:
obj_expr=sum(model.c[i]*model.x[i] for i in model.N)
model.obj=pe.Objective(sense=pe.maximize, expr=obj_expr)

In [34]:
con_lhs_expr = sum(model.a[i] * model.x[i] for i in model.N)
con_rhs_expr = model.b
model.con = pe.Constraint(expr=(con_lhs_expr <= con_rhs_expr))
result = solver.solve(model)

In [37]:
for i in model.N:
  print(pe.value(model.x[i]))
print(pe.value(model.obj))

1.0
0.0
1.0
1.0
1.0
25.0


In [36]:
model.pprint()

1 RangeSet Declarations
    N : Dimen=1, Size=5, Bounds=(1, 5)
        Key  : Finite : Members
        None :   True :   [1:5]

3 Param Declarations
    a : Size=5, Index=N, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :     2
          2 :     3
          3 :     4
          4 :     5
          5 :     9
    b : Size=1, Index=None, Domain=Any, Default=None, Mutable=False
        Key  : Value
        None :    20
    c : Size=5, Index=N, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :     3
          2 :     4
          3 :     5
          4 :     8
          5 :     9

1 Var Declarations
    x : Size=5, Index=N
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :   1.0 :     1 : False : False : Binary
          2 :     0 :   0.0 :     1 : False : False : Binary
          3 :     0 :   1.0 :     1 : False : False : Binary
          4 :     0 :   1.0 :     1 : False : False : Binary
          5 :     0 : 

## Shortestpath

In [ ]:
from collections import defaultdict
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from pyomo.environ import (ConcreteModel, Set, Param, Var, Objective, ConstraintList, Constraint, maximize, minimize, Binary, Reals)
from pyomo.opt import SolverFactory

In [ ]:
nodes = []
with open('nodes.txt') as fh:
  for line in fh:
    nodes.append(int(line.strip()))
nodes